# Handle Data Storage

### Create a list of all Zettel

In [1]:
from zettelsortierung.DataTypes import *
from zettelsortierung.Transformation import *
from zettelsortierung.RegionDetection import OpenCVRegionDetector
from zettelsortierung.OCR import OCR, OCRpreprocessing, OCRpostprocessing
from zettelsortierung.Visualisation import vis_boxes_labels

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
root = '../data/processed/Scans.parquet'
sammlung = Zettelsammlung.from_parquet(root, k=200)
print(len(sammlung))
sammlung = Zettelsammlung([scan for scan in sammlung if scan.id[-1] == '1'])
print(len(sammlung))

200
100


In [3]:

detector = OpenCVRegionDetector()

box_probe = Probe()
text_probe = Probe()

pipeline = \
Composition(
    Batch(1000),
    SequentialApp(
        ParallelApp(
            PatchDetect(detector)
        ),
        Flatten(),
        ParallelApp(
            BoundingBoxPad(10),
        ),
        Store(box_probe),
        ParallelApp(
            CutOutPatch(),
            ResizePatch(),
            BGR2RGB()
        ),
        Sort(key=lambda item: item.feature.shape[1]),
        Batch(16),
        ParallelApp(OCRpreprocessing()),
        SequentialApp(OCR()),
        ParallelApp(OCRpostprocessing()),
        Flatten(),
        Store(text_probe)
    ),
    Flatten()
)

In [4]:
res = pipeline(sammlung)

In [5]:
print(box_probe)
print(len(box_probe))
print(text_probe)
print(len(text_probe))

Probe([DataPoint(scan=Scan(A01-00000003_1, A01_a-Adel/1/01_a/), feature_id=0, feature=BoundingBox(x=np.int32(990), y=np.int32(262), w=np.int32(954), h=np.int32(128))),
DataPoint(scan=Scan(A01-00000115_1, A01_a-Adel/1/01_a/), feature_id=4, feature=BoundingBox(x=np.int32(1575), y=np.int32(1094), w=np.int32(251), h=np.int32(78))),
DataPoint(scan=Scan(A01-00000153_1, A01_a-Adel/1/02_Ao/), feature_id=0, feature=BoundingBox(x=np.int32(1513), y=np.int32(299), w=np.int32(342), h=np.int32(93))),
DataPoint(scan=Scan(A01-00000153_1, A01_a-Adel/1/02_Ao/), feature_id=1, feature=BoundingBox(x=np.int32(1500), y=np.int32(1071), w=np.int32(363), h=np.int32(89))),
DataPoint(scan=Scan(A01-00000053_1, A01_a-Adel/1/01_a/), feature_id=0, feature=BoundingBox(x=np.int32(1543), y=np.int32(1017), w=np.int32(234), h=np.int32(90))),
DataPoint(scan=Scan(A01-00000033_1, A01_a-Adel/1/01_a/), feature_id=0, feature=BoundingBox(x=np.int32(1447), y=np.int32(350), w=np.int32(155), h=np.int32(88))),
DataPoint(scan=Scan(A0

In [6]:
# Build lookup dictionary from second list
text_lookup = {(scan, feat_id): text for scan, feat_id, text in text_probe}

# Join
pairs = [
    (scan, feat_id, box, text_lookup[(scan, feat_id)])
    for scan, feat_id, box in box_probe
    if (scan, feat_id) in text_lookup
]

In [7]:
counter = 0
number = 10
zettel_set = set([dp.scan for dp in box_probe])
for scan in zettel_set:
    if counter >= number:
        break
    boxes = []
    labels = []
    for pair in pairs:
        if pair[0] == scan:
            boxes.append(pair[2])
            labels.append(pair[3])
    vis_boxes_labels(scan, boxes, labels)
    counter += 1
